# 02 - Landing -> Bronze

Lê os CSV do volume `workspace.landing.dados`, adiciona colunas de metadado e grava tabelas **Delta MANAGED** no schema `bronze`, sem alterar o conteúdo dos dados.

## display(dbutils.fs.ls('caminho'))

Mostra os arquivos que estão no volume `dados` (catálogo `workspace`, schema `landing`).

In [ ]:
display(dbutils.fs.ls('/Volumes/workspace/landing/dados/'))

## spark.read.option("inferSchema", "true").option("header", "true").csv("path")

Gera um DataFrame para cada arquivo do volume e adiciona duas colunas de metadado: data/hora do processamento (`datetime_bronze`) e nome do arquivo de origem (`filename`).

> Atenção: a opção correta é `inferSchema`. Escrita como `infeschema` o Spark ignora a opção em silêncio e **todas** as colunas chegam como string.

In [ ]:
from pyspark.sql import functions as F

landing_path = "/Volumes/workspace/landing/dados"

# tabela bronze -> arquivo de origem na landing
FILES = {
    "metacritic_games":  "games.csv",
    "steam_metrics":     "steam_metrics.csv",
    "regional_sales":    "regional_sales.csv",
    "how_long_to_beat":  "how_long_to_beat.csv",
    "walkthrough_links": "walkthrough_links.csv",
}

dataframes = {}
for table, filename in FILES.items():
    dataframes[table] = (
        spark.read.option("inferSchema", "true").option("header", "true")
             .csv(f"{landing_path}/{filename}")
             .withColumn("datetime_bronze", F.current_timestamp())
             .withColumn("filename", F.lit(filename))
    )

dataframes["metacritic_games"].show(5)
dataframes["metacritic_games"].printSchema()

## df.write.format('delta').saveAsTable("schema.tabela")

Salva os DataFrames como Delta Lake no schema `bronze`. As tabelas geradas são do tipo MANAGED (gerenciadas).

`overwriteSchema` permite regravar a tabela quando o schema muda (por exemplo, de string para tipos inferidos) sem precisar apagá-la antes.

In [ ]:
for table, df in dataframes.items():
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.bronze.{table}"))
    print(f"OK  workspace.bronze.{table}")

## (SQL) SHOW TABLES IN bronze

Verifica as tabelas Delta gravadas na camada bronze.

In [ ]:
%sql
SHOW TABLES IN bronze

## (SQL) DESCRIBE DETAIL nome_tabela;

Detalhes de uma tabela Delta Lake.

In [ ]:
%sql
DESCRIBE DETAIL bronze.metacritic_games;

## (SQL) DESCRIBE EXTENDED nome_tabela;

Mostra o tipo da tabela (MANAGED ou EXTERNAL) e o schema.

In [ ]:
%sql
DESCRIBE EXTENDED bronze.metacritic_games;